# Q3 - Association Rule Filtering (A/B/C)
A: support + confidence, B: + lift > 1, C: + p-value < 0.05

In [1]:
import warnings
from itertools import combinations

import pandas as pd
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)

In [2]:
url = "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_1000_onehot.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.strip()
df = df.astype(bool)

min_sup = 0.05
min_conf = 0.3
top_n = 30
print("Dataset shape:", df.shape)

Dataset shape: (1000, 30)


In [3]:
def run_compact(dataframe, min_support, max_k=6, top_n_keep=30):
    total = len(dataframe)
    bit_data = {}
    for col in dataframe.columns:
        bitstring = "".join(dataframe[col].astype(int).astype(str))
        bit_data[frozenset([col])] = int(bitstring, 2)

    levels = {1: {}}
    for itemset, bit in bit_data.items():
        support = bin(bit).count("1") / total
        if support >= min_support:
            levels[1][itemset] = (bit, support)

    levels[1] = dict(
        sorted(levels[1].items(), key=lambda x: x[1][1], reverse=True)[:top_n_keep]
    )

    for k in range(2, max_k + 1):
        levels[k] = {}
        prev = list(levels[k - 1].keys())
        candidates = set()

        for i in range(len(prev)):
            for j in range(i + 1, len(prev)):
                union = prev[i] | prev[j]
                if len(union) == k:
                    candidates.add(union)

        for c in candidates:
            if all((c - frozenset([x])) in levels[k - 1] for x in c):
                items = list(c)
                bit = bit_data[frozenset([items[0]])]
                for item in items[1:]:
                    bit &= bit_data[frozenset([item])]
                support = bin(bit).count("1") / total
                if support >= min_support:
                    levels[k][c] = (bit, support)

        if len(levels[k]) == 0:
            break

        levels[k] = dict(
            sorted(levels[k].items(), key=lambda x: x[1][1], reverse=True)[:top_n_keep]
        )

    out = {}
    for level in levels.values():
        for itemset, (_, support) in level.items():
            out[itemset] = support
    return out


def calc_p_value(dataframe, antecedent, consequent):
    x = dataframe[list(antecedent)].all(axis=1)
    y = dataframe[list(consequent)].all(axis=1)
    n11 = int(((x) & (y)).sum())
    n10 = int(((x) & (~y)).sum())
    n01 = int(((~x) & (y)).sum())
    n00 = int(((~x) & (~y)).sum())

    row1 = n11 + n10
    row2 = n01 + n00
    col1 = n11 + n01
    col2 = n10 + n00
    if row1 == 0 or row2 == 0 or col1 == 0 or col2 == 0:
        return 1.0

    _, p_value, _, _ = chi2_contingency([[n11, n10], [n01, n00]])
    return float(p_value)


def generate_rules(freq_itemsets, dataframe):
    rules = []
    for itemset, support_xy in freq_itemsets.items():
        if len(itemset) < 2:
            continue

        for i in range(1, len(itemset)):
            for antecedent_tuple in combinations(itemset, i):
                antecedent = frozenset(antecedent_tuple)
                consequent = itemset - antecedent

                support_x = freq_itemsets.get(antecedent, 0.0)
                support_y = freq_itemsets.get(consequent, 0.0)
                if support_x == 0.0 or support_y == 0.0:
                    continue

                confidence = support_xy / support_x
                lift = confidence / support_y
                p_value = calc_p_value(dataframe, antecedent, consequent)

                rules.append(
                    {
                        "antecedent": antecedent,
                        "consequent": consequent,
                        "support": support_xy,
                        "confidence": confidence,
                        "lift": lift,
                        "p_value": p_value,
                    }
                )

    return pd.DataFrame(rules)

In [4]:
freq_itemsets = run_compact(df, min_sup, max_k=6, top_n_keep=top_n)
rules = generate_rules(freq_itemsets, df)

rules_A = rules[(rules["support"] >= min_sup) & (rules["confidence"] >= min_conf)].copy()
rules_B = rules_A[rules_A["lift"] > 1].copy()
rules_C = rules_B[rules_B["p_value"] < 0.05].copy()

removed_A = rules[(rules["support"] < min_sup) | (rules["confidence"] < min_conf)].copy()
removed_B = rules_A[rules_A["lift"] <= 1].copy()
removed_C = rules_B[rules_B["p_value"] >= 0.05].copy()

print("All rules:", len(rules))
print("(A) Basic rules:", len(rules_A))
print("(B) Strong rules:", len(rules_B))
print("(C) Significant rules:", len(rules_C))

print("\nExample removed in A:")
if not removed_A.empty:
    print(removed_A.sort_values(["confidence", "support"]).head(1).to_string(index=False))
else:
    print("No rule removed in A from this generated rule set.")

print("\nExample removed in B:")
if not removed_B.empty:
    print(removed_B.sort_values("lift").head(1).to_string(index=False))
else:
    print("No rule removed in B.")

print("\nExample removed in C:")
if not removed_C.empty:
    print(removed_C.sort_values("p_value", ascending=False).head(1).to_string(index=False))
else:
    print("No rule removed in C.")

print("\nTop 10 significant rules (C):")
print(
    rules_C.sort_values(["lift", "confidence"], ascending=False)
    .head(10)
    .to_string(index=False)
)

All rules: 224
(A) Basic rules: 121
(B) Strong rules: 86
(C) Significant rules: 25

Example removed in A:
      antecedent               consequent  support  confidence    lift  p_value
frozenset({bag}) frozenset({straw, soda})    0.052    0.074605 0.88816 0.133529

Example removed in B:
              antecedent       consequent  support  confidence    lift  p_value
frozenset({straw, soda}) frozenset({bag})    0.052    0.619048 0.88816 0.133529

Example removed in C:
       antecedent       consequent  support  confidence     lift  p_value
frozenset({beer}) frozenset({bag})    0.173    0.697581 1.000833      1.0

Top 10 significant rules (C):
              antecedent               consequent  support  confidence     lift       p_value
      frozenset({chips})        frozenset({beer})    0.113    1.000000 4.032258  5.139122e-85
       frozenset({beer})       frozenset({chips})    0.113    0.455645 4.032258  5.139122e-85
frozenset({water, milk}) frozenset({bag, cereal})    0.056    0.417